# Cats vs Dogs - Image Classification with CNN
**Daily Challenge | DI Bootcamp**

## Roadmap
1. Data loading & generators
2. Data inspection
3. Model architecture
4. Optimization setup
5. Training
6. Validation evaluation
7. Test inference & CSV export
8. Baseline vs augmentation
9. Class imbalance handling
10. Save artifacts
11. Transfer learning extension


---
## Step 1 - Data Loading & Generators

**Setup**: download the [Cats vs Dogs dataset](https://www.microsoft.com/en-us/download/details.aspx?id=54765), extract it, rename the root folder to `cats_dogs` and place it inside `data/`.

Expected layout:
```
data/cats_dogs/train/train/cat.0.jpg
data/cats_dogs/test/test/1.jpg
```


In [ ]:
# Prefilled. Just copy and execute.
import os, re
from glob import glob
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split

np.random.seed(42); tf.random.set_seed(42)

DATA_ROOT = Path('data/cats_dogs')
train_dir = (DATA_ROOT/'train'/'train') if (DATA_ROOT/'train'/'train').exists() else (DATA_ROOT/'train')
test_dir  = (DATA_ROOT/'test'/'test')   if (DATA_ROOT/'test'/'test').exists()  else (DATA_ROOT/'test')

IMG_HEIGHT, IMG_WIDTH = 48, 48
BATCH_SIZE = 32
SEED = 1337

def build_df_from_folder(folder, labeled=True):
    exts = ('*.jpg','*.jpeg','*.png','*.bmp')
    files = []
    for ex in exts:
        files.extend(glob(str(folder/'**'/ex), recursive=True))
    if not files:
        raise FileNotFoundError(f'No images found under {folder}')
    rows = []
    for f in files:
        if labeled:
            name = Path(f).name.lower()
            parent = Path(f).parent.name.lower()
            if parent in {'cat','cats'}: label = 'cat'
            elif parent in {'dog','dogs'}: label = 'dog'
            else:
                if re.search(r'(^|[^a-z])cat([^a-z]|$)', name): label = 'cat'
                elif re.search(r'(^|[^a-z])dog([^a-z]|$)', name): label = 'dog'
                else: continue
            rows.append({'filepath': f, 'label': label})
        else:
            rows.append({'filepath': f})
    return pd.DataFrame(rows)

df_train_full = build_df_from_folder(train_dir, labeled=True)
df_test_full  = build_df_from_folder(test_dir,  labeled=False)

df_tr, df_val = train_test_split(
    df_train_full, test_size=0.2,
    stratify=df_train_full['label'], random_state=SEED
)

train_gen = ImageDataGenerator(
    rescale=1./255, rotation_range=45,
    width_shift_range=0.15, height_shift_range=0.15,
    zoom_range=0.5, horizontal_flip=True,
)
val_gen  = ImageDataGenerator(rescale=1./255)
test_gen = ImageDataGenerator(rescale=1./255)

train_flow = train_gen.flow_from_dataframe(
    df_tr, x_col='filepath', y_col='label',
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode='binary', batch_size=BATCH_SIZE,
    shuffle=True, seed=SEED, validate_filenames=False
)
val_flow = val_gen.flow_from_dataframe(
    df_val, x_col='filepath', y_col='label',
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode='binary', batch_size=BATCH_SIZE,
    shuffle=False, validate_filenames=False
)
test_flow = test_gen.flow_from_dataframe(
    df_test_full, x_col='filepath', y_col=None,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode=None, batch_size=BATCH_SIZE,
    shuffle=False, validate_filenames=False
)

print({'train': train_flow.samples, 'val': val_flow.samples,
       'test': test_flow.samples, 'class_indices': train_flow.class_indices})


---
## Step 2 - Data Inspection

### 2.1 Class balance


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

label_counts = df_train_full['label'].value_counts()
total = len(df_train_full)
print('CLASS BALANCE REPORT')
print('-' * 35)
for cls, cnt in label_counts.items():
    print(f'  {cls:>6}  {cnt:>5} images  ({cnt/total*100:.1f}%)')
print(f'  Total   {total:>5} images')
print('-' * 35)
gap = abs(label_counts.iloc[0] - label_counts.iloc[1]) / total
print('Classes BALANCED' if gap < 0.05 else 'Classes IMBALANCED - consider class_weight')


### 2.2 Sample image grid

**Visual cues that distinguish cats from dogs:**
- **Ear shape**: cats have pointed triangular ears; dogs have floppy or rounded ears
- **Snout**: dogs have longer protruding snouts; cats have flatter faces
- **Eye shape**: cats have almond-shaped eyes; dogs have rounder eyes
- **Fur texture**: cat fur is finer and more uniform; dog breeds vary widely


In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(10, 10))
fig.suptitle('Training Sample Grid (4x4)', fontsize=14, fontweight='bold')

images, labels = next(train_flow)
class_names = {v: k for k, v in train_flow.class_indices.items()}

for i, ax in enumerate(axes.flat):
    if i < len(images):
        ax.imshow(images[i])
        ax.set_title(class_names[int(labels[i])].upper(), fontsize=10,
                     color='#e74c3c' if int(labels[i]) == 0 else '#3498db',
                     fontweight='bold')
    ax.axis('off')
plt.tight_layout()
plt.show()


---
## Step 3 - Model Architecture

### Design rationale

The model has **3 convolutional blocks** (32->64->128 filters), each followed by MaxPooling2D(2,2) to halve spatial dimensions and introduce translational invariance.

**Dropout** (rate=0.4) before the dense layers prevents the model from over-relying on any single activation, forcing redundant feature learning and directly reducing overfitting.

**Output**: single sigmoid neuron outputting P(dog). Binary cross-entropy is the correct loss for this Bernoulli target (as opposed to softmax + categorical cross-entropy for multi-class).


In [ ]:
from tensorflow.keras import Sequential, Input
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

def build_model(input_shape=(IMG_HEIGHT, IMG_WIDTH, 3), dropout_rate=0.4):
    model = Sequential([
        Input(shape=input_shape),
        # Block 1
        Conv2D(32, (3,3), activation='relu', padding='same'),
        MaxPooling2D(2, 2),
        # Block 2
        Conv2D(64, (3,3), activation='relu', padding='same'),
        MaxPooling2D(2, 2),
        # Block 3
        Conv2D(128, (3,3), activation='relu', padding='same'),
        MaxPooling2D(2, 2),
        # Classifier head
        Flatten(),
        Dropout(dropout_rate),
        Dense(256, activation='relu'),
        Dropout(dropout_rate),
        Dense(1, activation='sigmoid')  # P(dog)
    ], name='CatsVsDogs_CNN')
    return model

model = build_model()
model.summary()


---
## Step 4 - Optimization Setup

| Hyperparameter | Value | Justification |
|---|---|---|
| Optimizer | Adam | Adaptive per-parameter LR; fast convergence; standard CNN baseline |
| Learning rate | 1e-4 | Conservative enough to not overshoot; fast enough for ~30 epochs |
| Loss | Binary cross-entropy | Correct for sigmoid + Bernoulli target |
| EarlyStopping | patience=5 | Stops training when val_loss stagnates; prevents overfitting |
| ReduceLROnPlateau | factor=0.5, patience=3 | Halves LR on plateau to escape local minima |
| Batch size | 32 | Stable gradients; fits in CPU/GPU RAM |


In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(monitor='val_loss', patience=5,
                           restore_best_weights=True, verbose=1)
reduce_lr  = ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                               patience=3, min_lr=1e-7, verbose=1)
checkpoint = ModelCheckpoint('best_model.h5', monitor='val_loss',
                             save_best_only=True, verbose=0)

print('Compilation done')
print(f'Total parameters: {model.count_params():,}')


---
## Step 5 - Model Training


In [ ]:
EPOCHS = 30

history = model.fit(
    train_flow,
    epochs=EPOCHS,
    validation_data=val_flow,
    callbacks=[early_stop, reduce_lr, checkpoint],
    verbose=1
)

print('Training complete')


### 5.1 Learning curves

**Detecting overfitting**: if training accuracy keeps rising while validation accuracy plateaus or drops, the model is memorizing the training set. The gap between train and val loss is the primary signal. Mitigations: stronger augmentation, higher dropout, fewer parameters.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Training & Validation Curves', fontsize=14, fontweight='bold')

axes[0].plot(history.history['accuracy'],     label='Train', color='#3498db', lw=2)
axes[0].plot(history.history['val_accuracy'], label='Val',   color='#e74c3c', lw=2, ls='--')
axes[0].set_title('Accuracy', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history.history['loss'],     label='Train', color='#3498db', lw=2)
axes[1].plot(history.history['val_loss'], label='Val',   color='#e74c3c', lw=2, ls='--')
axes[1].set_title('Loss', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Binary Cross-Entropy')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

final_train = history.history['accuracy'][-1]
final_val   = history.history['val_accuracy'][-1]
gap = final_train - final_val
print(f'Train accuracy : {final_train:.4f}')
print(f'Val   accuracy : {final_val:.4f}')
print(f'Gap            : {gap:.4f}', '-> Overfitting detected' if gap > 0.10 else '-> Reasonable')


---
## Step 6 - Validation Evaluation

**Error type interpretation**: False Negatives (dog predicted as cat) are more dangerous if we need high recall for dogs. False Positives (cat predicted as dog) are more costly if precision matters. The ROC curve and probability distribution guide threshold selection.
The default 0.5 threshold is arbitrary for sigmoid outputs and should be calibrated to the specific cost asymmetry of the task.


In [ ]:
from sklearn.metrics import (confusion_matrix, classification_report,
    ConfusionMatrixDisplay, roc_auc_score, roc_curve)

model.load_weights('best_model.h5')
val_flow.reset()
val_preds_proba = model.predict(val_flow, verbose=0).flatten()
val_preds       = (val_preds_proba >= 0.5).astype(int)
val_true        = val_flow.labels

val_loss, val_acc = model.evaluate(val_flow, verbose=0)
auc = roc_auc_score(val_true, val_preds_proba)

print('=' * 45)
print(f'  Validation Loss     : {val_loss:.4f}')
print(f'  Validation Accuracy : {val_acc:.4f}  ({val_acc*100:.1f}%)')
print(f'  ROC-AUC             : {auc:.4f}')
print('=' * 45)
print(classification_report(val_true, val_preds, target_names=['cat','dog']))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Validation Evaluation', fontsize=14, fontweight='bold')

cm = confusion_matrix(val_true, val_preds)
ConfusionMatrixDisplay(cm, display_labels=['cat','dog']).plot(
    ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Confusion Matrix', fontweight='bold')

axes[1].hist(val_preds_proba[val_true==0], bins=30, alpha=0.7,
             color='#e74c3c', label='Actual: cat', edgecolor='white')
axes[1].hist(val_preds_proba[val_true==1], bins=30, alpha=0.7,
             color='#3498db', label='Actual: dog', edgecolor='white')
axes[1].axvline(0.5, color='black', ls='--', lw=1.5, label='Threshold 0.5')
axes[1].set_title('Prediction Distribution', fontweight='bold')
axes[1].set_xlabel('P(dog)'); axes[1].legend()

fpr, tpr, _ = roc_curve(val_true, val_preds_proba)
axes[2].plot(fpr, tpr, color='#9b59b6', lw=2, label=f'AUC = {auc:.3f}')
axes[2].plot([0,1],[0,1],'k--', lw=1)
axes[2].set_title('ROC Curve', fontweight='bold')
axes[2].set_xlabel('False Positive Rate')
axes[2].set_ylabel('True Positive Rate')
axes[2].legend()

plt.tight_layout(); plt.show()


---
## Step 7 - Inference on Unlabeled Test Set

We export a CSV with `filepath`, `prob_dog`, and `pred_label`. **Manual sanity check**: display 20 random test images alongside their predicted label and spot-check for systematic errors (e.g., all white animals predicted as the same class).


In [ ]:
test_flow.reset()
test_proba = model.predict(test_flow, verbose=1).flatten()

THRESHOLD = 0.5
test_labels = ['dog' if p >= THRESHOLD else 'cat' for p in test_proba]

predictions_df = pd.DataFrame({
    'filepath'  : df_test_full['filepath'].values[:len(test_proba)],
    'prob_dog'  : test_proba.round(4),
    'pred_label': test_labels
})
predictions_df.to_csv('test_predictions.csv', index=False)
print(f'Saved {len(predictions_df)} predictions to test_predictions.csv')
print(predictions_df['pred_label'].value_counts())
predictions_df.head(10)


---
## Step 8 - Baseline vs Augmentation Comparison

We train the **same architecture** with no augmentation (rescale only) to isolate the impact of data augmentation on generalization.


In [ ]:
baseline_gen = ImageDataGenerator(rescale=1./255)
baseline_flow = baseline_gen.flow_from_dataframe(
    df_tr, x_col='filepath', y_col='label',
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode='binary', batch_size=BATCH_SIZE,
    shuffle=True, seed=SEED, validate_filenames=False
)

model_baseline = build_model()
model_baseline.compile(optimizer=Adam(1e-4),
                       loss='binary_crossentropy', metrics=['accuracy'])

print('Training baseline (no augmentation)...')
history_baseline = model_baseline.fit(
    baseline_flow, epochs=EPOCHS,
    validation_data=val_flow,
    callbacks=[EarlyStopping(monitor='val_loss', patience=5,
                             restore_best_weights=True, verbose=0)],
    verbose=1
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Augmented vs Baseline - Learning Curves', fontsize=14, fontweight='bold')

colors = {'aug': '#3498db', 'base': '#e74c3c'}
for ax, metric, title in zip(axes, ['accuracy','loss'], ['Accuracy','Loss']):
    ax.plot(history.history[metric],          label='Aug - Train',  color=colors['aug'],  lw=2)
    ax.plot(history.history[f'val_{metric}'], label='Aug - Val',    color=colors['aug'],  lw=2, ls='--')
    ax.plot(history_baseline.history[metric],          label='Base - Train', color=colors['base'], lw=2)
    ax.plot(history_baseline.history[f'val_{metric}'], label='Base - Val',   color=colors['base'], lw=2, ls='--')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Epoch'); ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()

_, aug_val_acc  = model.evaluate(val_flow, verbose=0)
_, base_val_acc = model_baseline.evaluate(val_flow, verbose=0)
print('=' * 50)
print(f'  Augmented  val accuracy : {aug_val_acc:.4f}')
print(f'  Baseline   val accuracy : {base_val_acc:.4f}')
print(f'  Improvement             : {(aug_val_acc-base_val_acc)*100:+.1f} pp')
print('=' * 50)
print('\nAnalysis:')
print('The baseline shows a wider train/val gap (overfitting): it memorizes exact images.')
print('Augmentation reduces this gap by exposing the model to varied transformations,')
print('forcing it to learn features robust to rotation, flip, and zoom.')


---
## Step 9 - Class Imbalance Handling

If classes are imbalanced, we weight the loss inversely to class frequency so the minority class contributes equally to the gradient. This improves recall for the minority class at a small cost to precision for the majority class.


In [ ]:
from sklearn.utils.class_weight import compute_class_weight

labels_array = df_tr['label'].map({'cat':0,'dog':1}).values
cw = compute_class_weight('balanced', classes=np.unique(labels_array), y=labels_array)
class_weight_dict = {int(k): float(v) for k, v in zip(np.unique(labels_array), cw)}
print('Class weights:', class_weight_dict)

ratio = max(cw) / min(cw)
if ratio > 1.2:
    print(f'Imbalance ratio {ratio:.2f} - retraining with class weights...')
    model_balanced = build_model()
    model_balanced.compile(optimizer=Adam(1e-4),
                           loss='binary_crossentropy', metrics=['accuracy'])
    model_balanced.fit(
        train_flow, epochs=EPOCHS, validation_data=val_flow,
        class_weight=class_weight_dict,
        callbacks=[EarlyStopping('val_loss', patience=5,
                                 restore_best_weights=True, verbose=0)],
        verbose=1
    )
    _, bal_acc = model_balanced.evaluate(val_flow, verbose=0)
    print(f'Balanced model val accuracy: {bal_acc:.4f}')
else:
    print(f'Imbalance ratio {ratio:.2f} - dataset balanced, no retraining needed')


---
## Step 10 - Save Artifacts

Saving both **weights** and **metadata** is essential for reproducibility: someone else (or you in 6 months) must be able to recreate the exact experiment from the config file and reload the model for inference without retraining.


In [ ]:
import json, datetime

model.save('cats_dogs_model.h5')
model.save('cats_dogs_saved_model')
print('Model saved')

config = {
    'timestamp'      : datetime.datetime.now().isoformat(),
    'img_size'       : [IMG_HEIGHT, IMG_WIDTH],
    'batch_size'     : BATCH_SIZE,
    'epochs_run'     : len(history.history['loss']),
    'optimizer'      : 'Adam',
    'learning_rate'  : 1e-4,
    'dropout_rate'   : 0.4,
    'augmentation'   : {'rotation':45,'shift':0.15,'zoom':0.5,'flip':True},
    'early_stopping_patience': 5,
    'final_val_accuracy'     : round(float(aug_val_acc), 4),
    'train_samples'          : train_flow.samples,
    'val_samples'            : val_flow.samples,
    'class_indices'          : train_flow.class_indices
}
with open('training_config.json','w') as f:
    json.dump(config, f, indent=2)
print('Config saved to training_config.json')
print(json.dumps(config, indent=2))


---
## Step 11 - Extension: Transfer Learning with MobileNetV2

**Justification**: MobileNetV2 was pre-trained on ImageNet (1.2M images, 1000 classes). It has already learned powerful low-level features (edges, textures) and mid-level patterns (shapes, parts) that transfer directly to cat/dog classification.

**Strategy**: freeze the backbone, train only the small classifier head. This gives strong accuracy in very few epochs with much less data.

**Expected benefit**: higher accuracy and better generalization than training from scratch, especially on smaller datasets.


In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import Model
from tensorflow.keras.layers import GlobalAveragePooling2D
import tensorflow as tf

def build_transfer_model(input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)):
    base = MobileNetV2(input_shape=input_shape, include_top=False, weights='imagenet')
    base.trainable = False
    inputs  = tf.keras.Input(shape=input_shape)
    x       = base(inputs, training=False)
    x       = GlobalAveragePooling2D()(x)
    x       = Dropout(0.3)(x)
    x       = Dense(128, activation='relu')(x)
    outputs = Dense(1, activation='sigmoid')(x)
    return Model(inputs, outputs, name='MobileNetV2_Transfer')

model_transfer = build_transfer_model()
model_transfer.compile(optimizer=Adam(1e-4),
                       loss='binary_crossentropy', metrics=['accuracy'])
model_transfer.summary()


In [ ]:
history_transfer = model_transfer.fit(
    train_flow, epochs=15, validation_data=val_flow,
    callbacks=[EarlyStopping('val_loss', patience=4,
                             restore_best_weights=True, verbose=0)],
    verbose=1
)
_, transfer_val_acc = model_transfer.evaluate(val_flow, verbose=0)
print('=' * 50)
print(f'  Scratch CNN   val accuracy : {aug_val_acc:.4f}')
print(f'  MobileNetV2   val accuracy : {transfer_val_acc:.4f}')
print(f'  Improvement                : {(transfer_val_acc-aug_val_acc)*100:+.1f} pp')
print('=' * 50)


---
## Step 12 - Deliverables Checklist

| # | Deliverable | Location |
|---|---|---|
| 1 | Class balance report + sample grid | Step 2 |
| 2 | Model architecture rationale | Step 3 |
| 3 | Optimization justification | Step 4 |
| 4 | Training curves + overfitting analysis | Step 5 |
| 5 | Confusion matrix, precision/recall, ROC-AUC | Step 6 |
| 6 | Test predictions CSV | `test_predictions.csv` |
| 7 | Augmented vs baseline comparison | Step 8 |
| 8 | Class imbalance analysis | Step 9 |
| 9 | Saved model files | `cats_dogs_model.h5`, `cats_dogs_saved_model/` |
| 10 | Training config | `training_config.json` |
| 11 | Transfer learning extension | Step 11 |

---
**Key reminders**:
- Only `train_flow` uses augmentation. `val_flow` and `test_flow` use rescale only.
- Keep the same `val_flow` across all experiments for consistent comparison.
- If results look wrong, recheck `train_flow.class_indices` first. Data bugs dominate model bugs.
- The 0.5 threshold is adjustable. Use the ROC curve (Step 6) to pick a threshold that
  matches the precision/recall tradeoff required by the task.
